# PlaceBot-first GeoNames alignment

This experiment treats PlaceBot as the georeferencing stage, then aligns its predicted coordinates to GeoNames identities. It uses PlaceBot's archived, pinned benchmark outputs so the experiment is reproducible without paid API calls.

The selected PlaceBot model can be changed below. GeoNames candidates are retrieved by names from the verbatim locality and ranked by distance from PlaceBot's predicted coordinates.

In [ ]:
from __future__ import annotations

import re
from functools import lru_cache

import numpy as np
import pandas as pd
import psycopg2
from tqdm.auto import tqdm

from pkb.config import settings
from pkb.models.places import search_geonames_candidates
from pkb.utils import country_string_to_iso


In [ ]:
PLACEBOT_COMMIT = "b036dd0b555d5ff32f7775bc0461707cc0df6285"
PLACEBOT_RAW = (
    "https://raw.githubusercontent.com/JackDanHollister/PlaceBot/"
    f"{PLACEBOT_COMMIT}/benchmarks/results"
)
PLACEBOT_MODEL = "gemini_2_5_pro"
SAMPLE_PER_DATASET = 50

benchmark_files = {
    "bombus_uk": f"{PLACEBOT_RAW}/bombus_uk_model_comparison.tsv",
    "odonata_global": f"{PLACEBOT_RAW}/odonata_global_model_comparison.tsv",
}

benchmarks = pd.concat(
    [
        pd.read_csv(url, sep="\t").head(SAMPLE_PER_DATASET).assign(benchmark=name)
        for name, url in benchmark_files.items()
    ],
    ignore_index=True,
)

latitude_column = f"latitude_{PLACEBOT_MODEL}"
longitude_column = f"longitude_{PLACEBOT_MODEL}"
distance_column = f"distance_km_{PLACEBOT_MODEL}"

benchmarks[["benchmark", "Barcode", "label_verbatim", "Country"]].head()


## GeoNames candidate retrieval

PlaceBot's archived comparison files contain predicted coordinates but not its parsed hierarchy. The transparent parser below extracts candidate name components from the original locality. This means the experiment tests whether a PlaceBot coordinate estimate can disambiguate name-based GeoNames candidates.

In [ ]:
DISTANCE_PREFIX = re.compile(
    r"^\s*\d+(?:\.\d+)?\s*(?:km|kms?|m|mi|miles?)\.?\s+"
    r"(?:north|south|east|west|n|s|e|w|ne|nw|se|sw)\.?\s+(?:of\s+)?",
    re.IGNORECASE,
)
LABEL_PREFIX = re.compile(r"^\s*\d+(?:st|nd|rd|th)\s+label\s*:\s*", re.IGNORECASE)


def locality_components(locality, country):
    country = str(country or "").strip().casefold()
    components = []
    for value in re.split(r"[,;:]", str(locality or "")):
        value = LABEL_PREFIX.sub("", value).strip(" .")
        value = DISTANCE_PREFIX.sub("", value).strip(" .")
        if len(value) >= 3 and value.casefold() != country:
            components.append(value)
    return list(dict.fromkeys(components))


In [ ]:
connection = psycopg2.connect(
    host=settings.postgres_host,
    port=settings.postgres_port,
    dbname=settings.postgres_database,
    user=settings.postgres_user,
    password=settings.postgres_password,
)


@lru_cache(maxsize=None)
def search_component(component, country_code):
    return search_geonames_candidates(
        connection,
        component,
        country_code=country_code,
        limit=5,
    )


def candidate_pool(row):
    country_code = country_string_to_iso(str(row.Country))
    candidates = {}
    for component in locality_components(row.label_verbatim, row.Country):
        for candidate in search_component(component, country_code):
            candidate = {**candidate, "query_component": component}
            current = candidates.get(candidate["geoname_id"])
            if current is None or (
                candidate["exact_match"], candidate["name_score"]
            ) > (current["exact_match"], current["name_score"]):
                candidates[candidate["geoname_id"]] = candidate
    return list(candidates.values())


In [ ]:
def haversine_km(latitude, longitude, reference_latitude, reference_longitude):
    latitude, longitude, reference_latitude, reference_longitude = np.radians(
        [latitude, longitude, reference_latitude, reference_longitude]
    )
    delta_latitude = reference_latitude - latitude
    delta_longitude = reference_longitude - longitude
    value = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(latitude)
        * np.cos(reference_latitude)
        * np.sin(delta_longitude / 2) ** 2
    )
    return 6371.0088 * 2 * np.arctan2(np.sqrt(value), np.sqrt(1 - value))


def evaluate_record(row):
    placebot_latitude = getattr(row, latitude_column)
    placebot_longitude = getattr(row, longitude_column)
    placebot_distance = getattr(row, distance_column)

    candidates = candidate_pool(row)
    if pd.notna(placebot_latitude) and pd.notna(placebot_longitude):
        candidates.sort(
            key=lambda candidate: haversine_km(
                placebot_latitude,
                placebot_longitude,
                candidate["latitude"],
                candidate["longitude"],
            )
        )
    else:
        candidates = []

    distances = [
        haversine_km(
            candidate["latitude"],
            candidate["longitude"],
            row.lat_manual,
            row.long_manual,
        )
        for candidate in candidates[:5]
    ]
    return {
        "placebot_distance_km": placebot_distance,
        "placebot_within_radius": (
            pd.notna(placebot_distance) and placebot_distance <= row.radius_km_manual
        ),
        "candidate_count": len(candidates),
        "top1_geoname_id": candidates[0]["geoname_id"] if candidates else None,
        "top1_distance_km": distances[0] if distances else np.nan,
        "top1_within_radius": bool(
            distances and distances[0] <= row.radius_km_manual
        ),
        "top5_within_radius": any(
            distance <= row.radius_km_manual for distance in distances
        ),
    }


In [ ]:
evaluations = [
    evaluate_record(row)
    for row in tqdm(benchmarks.itertuples(index=False), total=len(benchmarks))
]
results = pd.concat(
    [benchmarks.reset_index(drop=True), pd.DataFrame(evaluations)],
    axis=1,
)


In [ ]:
metrics = pd.Series(
    {
        "records": len(results),
        "placebot_resolved": results["placebot_distance_km"].notna().sum(),
        "placebot_within_radius": results["placebot_within_radius"].sum(),
        "placebot_accuracy": results["placebot_within_radius"].mean(),
        "placebot_precision": (
            results["placebot_within_radius"].sum()
            / results["placebot_distance_km"].notna().sum()
        ),
        "placebot_median_distance_km": results["placebot_distance_km"].median(),
        "geonames_resolved": results["candidate_count"].gt(0).sum(),
        "geonames_unresolved": results["candidate_count"].eq(0).sum(),
        "geonames_coverage": results["candidate_count"].gt(0).mean(),
        "geonames_top1_accuracy": results["top1_within_radius"].mean(),
        "geonames_top1_precision": (
            results["top1_within_radius"].sum()
            / results["candidate_count"].gt(0).sum()
        ),
        "geonames_top5_accuracy": results["top5_within_radius"].mean(),
        "geonames_median_top1_distance_km": results["top1_distance_km"].median(),
    }
)
metrics


In [ ]:
results.loc[
    ~results["top1_within_radius"],
    [
        "benchmark",
        "Barcode",
        "label_verbatim",
        "Country",
        "placebot_distance_km",
        "top1_distance_km",
        "candidate_count",
    ],
].head(25)


In [ ]:
connection.close()
